In [ ]:
import numpy as np
%load_ext autoreload
%autoreload 0

Notebook by **Maxime Dion** <maxime.dion@usherbrooke.ca><br>
For the QSciTech-QuantumBC virtual workshop on gate-based quantum computing

# Before you begin

Make sure you have completed the first tutorial.

## Tutorial 2 (Estimation)

**Important : Complete Tutorial 1 first**

In this tutorial you will complete the implementation of the functions in `estimation.py` and `vqe.py`. 

By completing all sections of this notebook you'll be able to :
- Prepare a Quantum State based on a varitional form (circuit);
- Measure qubits in the X, Y and Z basis;
- Estimate expectation value of `PauliString` on a quantum state;
- Evaluate the expectation value of an Hamiltonian in the form of a `Operator`;
- Run a minimization algorithm on the energy expectation fonction to find the ground state of a Hamiltonian;
- Dance to express your overwhelming sense of accomplishment

The solution we suggest here is NOT mandatory. If you find ways to make it better and more efficient, go on and impress us! 

**Remark on qiskit**

We use `qiskit` in this workshop. The tools you'll be building here are already available in `qiskit`. For instance, you could use an `Estimator` to estimate the expectation value of an observable. You could also use the VQE solution. Nevertheless, we strongly encourage you to complete the current implementation because we think it as a valuable learning experience.

# Variationnal Quantum States

Every quantum circuit starts with all qubits in the state $|0\rangle$. In order to prepare a quantum state $|\psi\rangle$ we need to prepare a `QuantumCircuit` that will modify the states of the qubits in order to get this specific state. The action of a circuit can always be represented as a unitiary operator.

\begin{align}
    |\psi\rangle &= \hat{U} |0 \ldots 0\rangle
\end{align}

For a parametric state the `QuantumCircuit` and therefore the unitary $U$ will depend on some parameters that we wirte as $\boldsymbol{\theta}$.

\begin{align}
    |\psi(\boldsymbol{\theta})\rangle &= \hat{U}(\boldsymbol{\theta}) |0 \ldots 0\rangle
\end{align}

We will see 2 ways to define Parametrized Quantum Circuits that represent Variationnal Quantum States. For the first method we only need the `QuantumCircuit` class from `qiskit.circuit`.

In [ ]:
from qiskit.circuit import QuantumCircuit

## Generating function
The easiest way to generate a parametrized `QuantumCircuit` is to implement a function that takes parameters as arguments and returns a `QuantumCircuit`. Here is such a function that generates a 2 qubits QuantumCircuit.

In [1]:
def example_2qubits_2params_quantum_circuit(theta, phi):

    qc = QuantumCircuit(2)
    qc.ry(theta,0)
    qc.rz(phi,0)
    qc.cx(0,1)
    
    return qc

To visualize this circuit we first need to call the generating function with dummy argument values for it to return a circuit. We can draw the circuit. The `'mpl'` option draws the circuit in a fancy way using `matplotlib`. If you are experiencing problems, you can remove this option.

In [ ]:
varform_qc = example_2qubits_2params_quantum_circuit
qc = varform_qc(1,2)
qc.draw('mpl')

## Using qiskit parameter

The other way to generate a parametrized `QuantumCircuit` is to use the `Parameter` class in `qiskit`.

In [ ]:
from qiskit.circuit import Parameter

Here is the same circuit as before done with this method.

In [ ]:
a = Parameter('a')
b = Parameter('b')
varform_qc = QuantumCircuit(2)
varform_qc.ry(a,0)
varform_qc.rz(b,0)
varform_qc.cx(0,1)

Done this way the parametrized circuit can be drawn right away.

In [ ]:
varform_qc.draw('mpl')

To see what are the parameters of a parametrized `QuantumCircuit` you can use

In [ ]:
varform_qc.parameters

To assign values to the different parameters we need to use the `QuantumCircuit.assign_paremeters()` method. This methods takes a `dict` as an argument containing the `Parameter`s and their `value`s.

In [ ]:
param_dict = {a : 1, b : 2}
qc = varform_qc.assign_parameters(param_dict)
qc.draw('mpl')

If you want to provide the parameter values as a `list` or a `np.array` you can build the `dict` directly. Just make sure that the order you use in `param_values` corresponds to the other of `varform_qc.parameters`.

In [ ]:
param_values = [1, 2]
param_dict = dict(zip(varform_qc.parameters,param_values))
print(param_dict)

## Varforms circuits for H2
Using the method of you choice, prepare 2 different 4-qubit `QuantumCircuit`s. 
- The first should take 1 parameter to cover the real coefficients state sub space spanned by $|0101\rangle$ and $|1010\rangle$.
- The second should take 3 parameters to cover the real coefficients state sub space spanned by $|0101\rangle$, $|0110\rangle$, $|1001\rangle$ and $|1010\rangle$.

Revisit the presentation to find such circuits.

In [ ]:
varform_4qubits_1param = QuantumCircuit(4)
a = Parameter('a')
"""
Your code here
"""

varform_4qubits_1param.draw('mpl')

In [ ]:
varform_4qubits_3params = QuantumCircuit(4)
a = Parameter('a')
b = Parameter('b')
c = Parameter('c')
"""
Your code here
"""

varform_4qubits_3params.draw('mpl')

# Estimation

The goal of the estimation is to estimate the expectation value of an observable for a given quantum state. 

## Pauli Based Measurements

Since our observable (the Hamiltonian) is given as a linear combination of Pauli strings, we will need to estimate the expectation value of these Pauli string.

\begin{align*}
    \langle \psi |\hat{O} | \psi \rangle 
    =
    \sum_i h_i  \langle\psi |\hat{P}_i | \psi \rangle.
\end{align*} 

### Diagonalisation

We have seen that even if a quantum computer can only measure qubits in the Z-basis, the X and Y-basis are accessible if we *rotate* the quantum state before measuring. 

\begin{align*}
    \langle\psi |\hat{P} | \psi \rangle 
    =
    \langle\psi |\hat{U}^\dagger_\text{diag} \hat{\mathcal{Z}} \hat{U}_\text{diag} | \psi \rangle 
    = 
    \langle\psi^\prime |\hat{\mathcal{Z}} | \psi^\prime \rangle.
\end{align*} 

Implement the function : `diagonal_pauli_with_circuit` in the `estimation.py` file. This should return a diagonal `PauliString` and the `QuantumCircuit` that performs a transformation which diagonalize it.

Test your code with the next cell.

In [ ]:
%autoreload
from quantum_chemistry.pauli import PauliString
from quantum_chemistry.estimation import diagonal_pauli_with_circuit

pauli_string = PauliString.from_str('ZIXY')
diagonal_pauli, diagonal_circuit,  = diagonal_pauli_with_circuit(pauli_string)
print(diagonal_pauli) #should be 'ZIZZ'
diagonal_circuit.draw('mpl')

This diagonalisation circuit should then be composed with the circuit which prepares the state $| \psi \rangle$ to get a new circuit which prepares the state $| \psi^\prime \rangle$. This new state will be sampled, meaning it will be measured many times. Each time it will return a bit string, one bit per qubit.

### Diagonal PauliString eigenvalue

Each bit string is an eigenstate to the diagonal Pauli string and there is a associated eigenvalue. This eigenvalue can only be `+1` or `-1`. Therefore you need to code the function `diagonal_pauli_eigenvalue` to compute this value.

\begin{align*}
    \hat{P} |q\rangle = \Lambda_q^{(P)} |q\rangle
\end{align*}

Use the following code to test you implementation. You should get `1`, `-1`, `1` and `-1`.

In [ ]:
%autoreload
from quantum_chemistry.estimation import diagonal_pauli_eigenvalue, bitstring_to_bits
from quantum_chemistry.pauli import PauliString

diag_pauli = PauliString.from_str("ZZZI")
print(diagonal_pauli_eigenvalue(diag_pauli,bitstring_to_bits("0001")))
print(diagonal_pauli_eigenvalue(diag_pauli,bitstring_to_bits("0100")))
print(diagonal_pauli_eigenvalue(diag_pauli,bitstring_to_bits("1100")))
print(diagonal_pauli_eigenvalue(diag_pauli,bitstring_to_bits("1110")))

### Diagonal PauliString expectation value

Let's now estimate the expectation value of a single diagonal `PauliString`. This needs to be done in the method `diagonal_pauli_expectation_value()`. Implement this method using the one you just implemented (`diagonal_pauli_eigenvalue`) and the following equation
\begin{align*}
    \langle\psi| \hat{P}|\psi \rangle \approx \frac{1}{N_\text{tot}}\sum_{q} N_q \Lambda_q^{(P)}
\end{align*}
where $\Lambda_q^{(P)}$ is the eigenvalue of the `PauliString` for a state $|q\rangle$ and $N_q$ is the counts, i.e. the number of times this state was measured. Testing your implementation, you should get an expectation value of `0.5`.

In [ ]:
%autoreload
from quantum_chemistry.estimation import diagonal_pauli_expectation_value
from quantum_chemistry.pauli import PauliString

diagonal_pauli = PauliString.from_str('ZIZZ')
counts = {'0110' : 25, '1001' : 75}
pauli_string_expectation_value = diagonal_pauli_expectation_value(diagonal_pauli, counts)
pauli_string_expectation_value

### Assemble circuits

There is one diagonalisation circuit for each `PauliString`. 

\begin{align*}
    \langle\psi |\hat{P}_i | \psi \rangle = \langle\psi |\hat{U}^{(i)\dagger}_{\text{diag}} \hat{\mathcal{Z}}_i \hat{U}^{(i)}_{\text{diag}} | \psi \rangle = \langle\psi^{(i)} | \hat{\mathcal{Z}}_i  | \psi^{(i)} \rangle.
\end{align*}

We need to construct the rotated state for each Pauli string and then apply the *measurements*. Given a list of `PauliString`s and a quantum circuit, this is the task of the function `prepare_estimation_circuits_and_diagonal_paulis`. 

In [ ]:
%autoreload
from quantum_chemistry.estimation import prepare_estimation_circuits_and_diagonal_paulis
from qiskit.circuit import QuantumCircuit

paulis = [PauliString.from_str('ZXZX'), PauliString.from_str('XZXZ')]

state_circuit = QuantumCircuit(4)
state_circuit.x([0,2])

estimation_circuits, diagonal_paulis = prepare_estimation_circuits_and_diagonal_paulis(paulis, state_circuit)

for estimation_circuit in estimation_circuits:
    print(estimation_circuit)

### Running the circuits

We need to run and sample these circuits using a `Backend` which could be a simulator or a quantum computer.  Here we provide a simple way to do that using the `AerSimulator`. It should be pretty straight forward to use a quantum computer instead.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator

backend = AerSimulator()

sampler = Sampler(mode=backend)
pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pass_manager.run(estimation_circuits)
job = sampler.run(isa_circuits)
results = job.result()

To access the sampling results for each circuit, you can use the index of the circuit in the list. Note that `meas` is the name of the classical register involved in the measurements of the qubits.

In [ ]:
for i in range(len(estimation_circuits)):
    counts = results[i].data.meas.get_counts()
    print(counts)

### Estimate the expectation value of multiple Pauli strings

Now that you can sample circuit, you can assemble the previous functions to compute the estimations of the expectation values for a list of `PauliString`s given a quantum state.

In [ ]:
from quantum_chemistry.estimation import estimate_paulis_expectation_values
from quantum_chemistry.pauli import PauliString
from qiskit.circuit import QuantumCircuit
from qiskit_aer import AerSimulator

backend = AerSimulator()

paulis = [PauliString.from_str('ZXZX'), PauliString.from_str('XZXZ')]

state_circuit = QuantumCircuit(4)
state_circuit.x([0,2])
state_circuit.h([1,3])

expectation_values = estimate_paulis_expectation_values(paulis, state_circuit, backend)

print(expectation_values)

### Estimate the expectation value of an Operator

This should be easy to do now.

In [ ]:
from quantum_chemistry.estimation import estimate_observable_expectation_value
from quantum_chemistry.pauli import PauliString, Operator
from qiskit.circuit import QuantumCircuit
from qiskit_aer import AerSimulator

backend = AerSimulator()

paulis = [PauliString.from_str('ZXZX'), PauliString.from_str('XZXZ')]
operator = Operator([1,-1],paulis)

state_circuit = QuantumCircuit(4)
state_circuit.x([0,2])
state_circuit.h([1,3])

expectation_values = estimate_observable_expectation_value(operator, state_circuit, backend)

print(expectation_values)

## The Hamiltonian evaluation test

We will now use tools from the previous tutorial to construct an Hamiltonian and estimate its expectation value for the basis quantum states $|0101\rangle$ and $|1010\rangle$ which should be respectively around `-1.82` and `-0.26`.

In [ ]:
from quantum_chemistry.estimation import estimate_observable_expectation_value
from quantum_chemistry.mapping import build_qubit_hamiltonian, creation_annihilation_operators_with_jordan_wigner
from quantum_chemistry.molecule.h2_molecule import load_h2_spin_orbital_integral
from qiskit.circuit import QuantumCircuit
from qiskit_aer import AerSimulator

backend = AerSimulator()

distance, one_body, two_body, nuc_eneg = load_h2_spin_orbital_integral("../h2_data","h2_mo_integrals_d_0750.npz")
creation_operators, annihilation_operators = creation_annihilation_operators_with_jordan_wigner(4)

qubit_hamiltonian = build_qubit_hamiltonian(one_body, two_body, creation_operators, annihilation_operators)

state_circuit = QuantumCircuit(4)
# state_circuit.x([0,2])
state_circuit.x([1,3])

expectation_values = estimate_observable_expectation_value(qubit_hamiltonian, state_circuit, backend)

print(expectation_values)

# Variationnal Quantum Eigensolver

In a final step we need to implement a solver that will explore the quantum state space to try to find the quantum state which minimize the energy.

Like any minimzation process this solver will need a couple of ingredients :
- A function to minimize : we will compute the expectation of the Hamiltonian.
- A method : an algorithm that generaly takes in a function and a set of starting parameters and returns the best guess for the optimal parameters that correspond to the minimal value of the function to minimize.
- A set of starting parameters.

### Cost function

The cost function need to compute the energy for a given set of parameter.

\begin{align*}
    E_0 
    = \min_{\boldsymbol{\theta}} E(\boldsymbol{\theta}) 
    = \min_{\boldsymbol{\theta}} \langle \psi(\boldsymbol{\theta}) | \hat{H} | \psi(\boldsymbol{\theta}) \rangle
\end{align*}

This function could be defined in the following way, provided that the `ansatz_circuit` and the `hamiltonian` are alreayd defined.

In [ ]:
def cost_function(params):
    
    state_circuit = ansatz_circuit.assign_parameters(params)
    hamiltonian_expectation_value = estimate_observable_expectation_value(hamiltonian, state_circuit, backend)

    return hamiltonian_expectation_value.real


### Minimizer

A minimizer that works OK for the VQE algorithm is the  Sequential Least SQuares Programming (SLSQP) algorithm. It's available in the `minimize` sub-module of [scipy](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html). We will make a Lambda function with the minimizer so we can set all sorts of parameters before feeding it to the solver.

In [ ]:
from scipy.optimize import minimize

minimizer = lambda cost_fun, starting_params : minimize(
    cost_fun,
    starting_params,
    method = 'SLSQP', 
    options = {'maxiter' : 5,'eps' : 1e-1, 'ftol' : 1e-4, 'disp' : True, 'iprint' : 2})

The `minimizer` now takes only 2 arguments : the function and the starting parameter's values. We also specify some options :
- A small value for the maximum number of iterations. You will find that running the VQE algorithm is expensive because of the `cost_function`. Either it's long to simulate on the simulator or because it's running on an actual quantum computer.
- A `eps` of `0.1`. This is the size of the step the algorithm is going to change the values of the parameters to try to estimate the slope of the function. By the way, a lot of minimizing algorithms use the slope of the function to know in which direction is the minimum. Since our parameters are all angles in radians a value of 0.1 seems reasonnable. Play with this value if you like.
- A `ftol` value of `1e-4`. This is the goal for the precision of the value of the minimum value. The chemical accuracy is around 1 milli-Hartree.
- We set `iprint` to `2` so to see what is going on. For your final implementation you can set this to `0`.

### Just a function

Now you can put these things together to implement the `h2_ansatz_circuit` and `minimize_expectation_value` functions. These functions should run the minimization process and return you the minimization results. From which you'll be able to extract the optimal parameters and the minimal energy associated with. We suggest to use the `varform_4qubits_1param` parametrized circuit for this task. 

In [ ]:
from quantum_chemistry.vqe import h2_ansatz_circuit, minimize_expectation_value

ansatz_circuit = h2_ansatz_circuit()

minimization_result = minimize_expectation_value(qubit_hamiltonian, ansatz_circuit, backend, minimizer)

opt_params = minimization_result.x
opt_value = minimization_result.fun
print(opt_params)
print(opt_value)

In [ ]:
print('Ground state position estimate (vqe) : ', opt_params)
print('Ground state energy estimate (electronic, vqe) : ', opt_value)
print('Ground state energy estimate (molecular, vqe) : ', opt_value + nuc_eneg)

## Exact Solution

If you want to compare the value you get with the VQE algorithm it would be nice to have the exact value. If you were able to implement the `to_matrix()` method for `PauliString` and `Operator` then you can find the exact value of the ground state. All you need is to diagonalise the matrix reprensenting the whole Hamiltonian and find the lowest eigenvalue! Obviously this will not be possible to do for very large systems.

In [ ]:
import numpy as np
hamiltonian_matrix = qubit_hamiltonian.to_matrix()
eig_values, eig_vectors = np.linalg.eigh(hamiltonian_matrix)
eig_order = np.argsort(eig_values)
eig_values = eig_values[eig_order]
eig_vectors = eig_vectors[:,eig_order]
ground_state_value, ground_state_vector = eig_values[0], eig_vectors[:,0]
print('Ground state vector (exact) : ', ground_state_vector)
print('Ground state energy (electronic, exact) : ', ground_state_value)
print('Ground state energy (molecular, exact) : ', ground_state_value + nuc_eneg)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig,ax = plt.subplots(1,1)
i_max = np.argmax(np.abs(ground_state_vector))
state = ground_state_vector * np.sign(ground_state_vector[i_max])
ax.bar(range(len(state)),np.abs(state),color=(np.real(state) > 0).choose(['r','b']))
plt.xticks(range(len(state)),[f"{i:04b}" for i in range(len(state))], size='small',rotation=60);

# What's next?

Now that you can find the ground state for a specific H2 molecule configuration (`d = 0.750`), you should be able to do that for many configurations, say `d = 0.3` to `2.0`. Doing that will enable you to plot the so-called dissociation curve : energy vs distance. Do not forget to include the Coulomb repulsion energy of the nucleus!

You could also run your algorithm on a noisy backend, either a noisy simulator or a real quantum computer. You've already seen on day 1 how to set/get a noisy backend. You'll see that noise messes things up pretty bad.

Running on real machine will introduce the problem of the qubit layout. You might want to change the `initial_layout` in the `execute_opts` so that your `varform` is not applying CNOT gates between qubits that are not connected. You know this needs to insert SWAP gates and this introduces more noise. Also covered in day 1.

To limit the effect of readout noise, you could add a `measure_filter` to your `evaluator`, so that each time you execute the `eval_circuits` you apply the filter to the results. Also covered in day 1.

Implement the simulatneous evaluation for bitwise commuting cliques or even for general commuting cliques.

Notebook by **Maxime Dion** <maxime.dion@usherbrooke.ca><br>
For the QSciTech-QuantumBC virtual workshop on gate-based quantum computing